In [1]:
# ingestion_db.py script ingests all 6 raw csv dataset files as it is into MySQL DB. 
# So that we can fetch the data from db into pandas, understand it and perform data cleaning & feature engineering on it.
%run ingestion_db.py

2026-09-26 11:20:25,729 - INFO - ========= Starting ingestion pipeline =========
2026-09-26 11:20:26,053 - INFO - Database 'vendor_data' is ready.
2026-09-26 11:20:26,057 - INFO - Found 6 CSV file(s) in 'data': ['begin_inventory.csv', 'end_inventory.csv', 'purchases.csv', 'purchase_prices.csv', 'sales.csv', 'vendor_invoice.csv']
2026-09-26 11:20:26,060 - INFO - Loading 'begin_inventory.csv' (16.6MB)...
2026-09-26 11:20:26,063 - INFO -   Using pandas streaming...
2026-09-26 11:20:26,065 - INFO - Reading 'begin_inventory' in 50,000-row chunks...
2026-09-26 11:21:47,411 - INFO - Loaded 'begin_inventory': 206,529 rows in 81.35s (pandas streaming)
2026-09-26 11:21:47,416 - INFO -   Creating indexes for 'begin_inventory'...
2026-09-26 11:21:47,423 - INFO - Successfully loaded 'begin_inventory' completely.
2026-09-26 11:21:47,425 - INFO - Loading 'end_inventory.csv' (18.1MB)...
2026-09-26 11:21:47,428 - INFO -   Using pandas streaming...
2026-09-26 11:21:47,431 - INFO - Reading 'end_inventory

In [2]:
import pandas as pd
import numpy as np
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# creating database connection
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

In [3]:
# checking tables present in the database
tables = pd.read_sql_query("SHOW TABLES", engine)
tables

,Tables_in_vendor_data
0,begin_inventory
1,end_inventory
2,purchase_prices
3,purchases
4,sales
5,vendor_invoice


In [4]:
for table in tables['Tables_in_vendor_data']:
    print('-'*50, f'{table}', '-'*50)
    print('Count of records:', pd.read_sql(f"select count(*) as cnt from {table}", engine)['cnt'].values[0])
    display(pd.read_sql(f"select * from {table} limit 5", engine))

-------------------------------------------------- begin_inventory --------------------------------------------------
Count of records: 206529


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,2024-01-01
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,2024-01-01
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,2024-01-01
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,2024-01-01
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,2024-01-01


-------------------------------------------------- end_inventory --------------------------------------------------
Count of records: 224489


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,2024-12-31
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,2024-12-31
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,2024-12-31
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,2024-12-31
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,2024-12-31


-------------------------------------------------- purchase_prices --------------------------------------------------
Count of records: 12261


...

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,69_MOUNTMEND_8412,69,8412,Tequila Ocho Plata Fresno,750mL,105,ALTAMAR BRANDS LLC,8124,2023-12-21,2024-01-02,2024-01-04,2024-02-16,35.71,6,214.26,1
1,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
2,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
3,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
4,76_DONCASTER_2034,76,2034,Glendalough Double Barrel,750mL,388,ATLANTIC IMPORTING COMPANY,8169,2023-12-24,2024-01-02,2024-01-09,2024-02-16,21.32,5,106.60,1


-------------------------------------------------- sales --------------------------------------------------
Count of records: 12825463


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-01,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,2024-01-02,750.0,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,2024-01-03,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,2024-01-08,750.0,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,2024-01-09,375.0,1,0.79,12546,JIM BEAM BRANDS COMPANY


-------------------------------------------------- vendor_invoice --------------------------------------------------
Count of records: 5543


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,2024-01-04,8124,2023-12-21,2024-02-16,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,2024-01-09,8169,2023-12-24,2024-02-16,5,106.60,4.61,None
3,480,BACARDI USA INC,2024-01-12,8106,2023-12-20,2024-02-05,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,2024-01-07,8170,2023-12-24,2024-02-12,1935,15527.25,429.20,None


In [5]:
purchases = pd.read_sql_query("select * from purchases where VendorNumber = 4466", engine)
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-02,2024-01-07,2024-02-21,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,2023-12-22,2024-01-01,2024-01-07,2024-02-21,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-07,2024-01-19,2024-02-26,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,2023-12-27,2024-01-05,2024-01-19,2024-02-26,9.41,6,56.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2187,81_PEMBROKE_5215,81,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-29,2025-01-04,2025-02-10,9.41,6,56.46,1
2188,62_KILMARNOCK_5255,62,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.35,5,46.75,1
2189,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-28,2025-01-04,2025-02-10,9.41,5,47.05,1
2190,6_GOULCREST_5215,6,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,13595,2024-12-20,2024-12-31,2025-01-04,2025-02-10,9.41,6,56.46,1


In [6]:
purchase_prices = pd.read_sql_query("select * from purchase_prices where VendorNumber = 4466", engine)
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [7]:
vendor_invoice = pd.read_sql_query("select * from vendor_invoice where VendorNumber = 4466", engine)
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,2024-01-07,8137,2023-12-22,2024-02-21,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,2024-01-19,8207,2023-12-27,2024-02-26,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,2024-01-18,8307,2024-01-03,2024-02-18,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,2024-01-27,8469,2024-01-14,2024-03-11,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,2024-02-04,8532,2024-01-19,2024-03-15,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,2024-02-09,8604,2024-01-24,2024-03-15,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,2024-02-17,8793,2024-02-05,2024-04-02,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,2024-03-01,8892,2024-02-12,2024-03-28,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,2024-03-07,8995,2024-02-19,2024-04-02,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,2024-03-12,9033,2024-02-22,2024-04-16,147,1377.87,6.61,None


In [8]:
sales = pd.read_sql_query("select * from sales where VendorNo = 4466", engine)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-12,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-15,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-01-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9448,9_BLACKPOOL_5215,9,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,2024-12-21,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9449,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-02,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9450,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-09,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
9451,9_BLACKPOOL_5255,9,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,2024-12-23,1750.0,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


- The purchases table contains actual purchase data, including the date of purchase, products (brands) purchased by vendors, the amount paid (in dollars), and the quantity purchased.
- The purchase_prices table provides product-wise purchase prices which acts as a product master table.
- The vendor_invoice table aggregates data from the purchases table, summarizing quantity and dollar amounts, along with an additional column for freight. This table maintains uniqueness based on vendor and PO number.
- The sales table captures actual sales transactions, detailing the brands purchased by vendors, the quantity sold, the selling price, and the revenue earned.

# Data Cleaning (SQL)

All cleaning is done via raw SQL queries executed against MySQL.
Pandas is used only to: (1) read tables from DB, (2) write cleaned tables back to DB.

Common steps (drop/rename columns, casts, VendorName trim, VendorCanonicalName,
BottleSizeCategory) are implemented once as reusable functions and called per table (DRY).

**Run order:** Top to bottom, once. Each section alters real tables, so re-running a
section after it has already succeeded will error on the rename/add-column steps.

## 0. Setup

In [9]:
def run_sql(query):
    """Execute a SQL query and return a DataFrame (for SELECT) or None (for DDL/DML)."""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        conn.commit()
        try:
            return pd.DataFrame(result.fetchall(), columns=result.keys())
        except:
            return None

def show(query):
    """Print result of a SELECT query."""
    df = run_sql(query)
    print(df.to_string(index=False))
    return df

## 0.1 Common cleaning functions

These steps are identical across multiple tables, so they're written once and
reused: dropping/renaming columns, casting types, trimming + canonicalizing
`VendorName`, and deriving `BottleSizeCategory`, `StockStatus`.

In [10]:
BOTTLE_SIZE_CASE = """
    CASE
        WHEN BottleSize IN ('50mL','50mL 3 Pk','50mL 4 Pk','50mL 5 Pk','50mL 12 Pk',
                            '100mL','100mL 4 Pk','20mL 5 Pk','5.0 Oz','22.0 Oz')
             THEN 'Mini'
        WHEN BottleSize IN ('150mL','162.5mL','180mL','187mL','187mL 2 Pk','187mL 3 Pk',
                            '187mL 4 Pk','200mL','200mL 3 Pk','200mL 4 Pk','200mL 5 Pk',
                            '250mL','250mL 4 Pk','300mL','330mL','375mL','375mL 2 Pk',
                            '375mL 3 Pk','3/100mL')
             THEN 'Half Bottle'
        WHEN BottleSize IN ('500mL','560mL','600mL','650mL','720mL','750mL','750mL 2 Pk',
                            '750mL 3 Pk','750mL 4 Pk','750mL 6 Pk','750mL 12 P',
                            '750mL  3','750mL + 1/','750mL + 2/','750mL + 3/','750mL + 4/',
                            '400mL','Liter','25.0','25','5/2 oz')
             THEN 'Standard'
        WHEN BottleSize IN ('1000mL','1000mL 2 Pk','1100ml','1500mL','1500ml',
                            '1750mL','1.5L','1.75L')
             THEN 'Large'
        WHEN BottleSize IN ('3000mL','3750mL','3L','4000mL','4L','5000mL','5L',
                            '6000mL','6L','9000mL','9L','18000mL','18L','19500mL',
                            '19.5L','20000mL','20L','128.0 Gal')
             THEN 'Bulk'
        ELSE 'Unknown'
    END
"""

VENDOR_CANONICAL_CASE = """
    CASE
        WHEN VendorName IN ('VINEYARD BRANDS INC', 'VINEYARD BRANDS LLC')
             THEN 'VINEYARD BRANDS LLC'
        WHEN VendorName IN ('SOUTHERN WINE & SPIRITS NE', 'SOUTHERN GLAZERS W&S OF NE')
             THEN 'SOUTHERN GLAZERS W&S OF NE'
        ELSE VendorName
    END
"""

def drop_columns(table, cols):
    """Drop one or more columns from a real table."""
    clauses = ", ".join(f"DROP COLUMN {c}" for c in cols)
    run_sql(f"ALTER TABLE vendor_data.{table} {clauses}")
    print(f'{table}: dropped columns {cols}')

def rename_columns(table, mapping):
    """Rename columns in place, e.g. {'Brand': 'ProductID'}."""
    for old, new in mapping.items():
        run_sql(f"ALTER TABLE vendor_data.{table} RENAME COLUMN {old} TO {new}")
    print(f'{table}: renamed columns {mapping}')

def cast_columns(table, casts):
    """Change column data types in place, e.g. {'SalesPrice': 'DECIMAL(10,2)'}."""
    for col, dtype in casts.items():
        run_sql(f"ALTER TABLE vendor_data.{table} MODIFY COLUMN {col} {dtype}")
    print(f'{table}: cast columns {casts}')

def add_bottle_size_category(table):
    """Add + populate BottleSizeCategory using the shared bucket logic."""
    run_sql(f"ALTER TABLE vendor_data.{table} ADD COLUMN BottleSizeCategory VARCHAR(50)")
    run_sql(f"UPDATE vendor_data.{table} SET BottleSizeCategory = {BOTTLE_SIZE_CASE}")
    print(f'{table}: BottleSizeCategory added.')

def add_vendor_canonical_name(table):
    """Trim VendorName, then add + populate VendorCanonicalName."""
    run_sql(f"UPDATE vendor_data.{table} SET VendorName = TRIM(VendorName)")
    run_sql(f"ALTER TABLE vendor_data.{table} ADD COLUMN VendorCanonicalName VARCHAR(255)")
    run_sql(f"UPDATE vendor_data.{table} SET VendorCanonicalName = {VENDOR_CANONICAL_CASE}")
    print(f'{table}: VendorName trimmed, VendorCanonicalName added.')

def add_stock_status(table):
    """Add + populate StockStatus from StockQuantity."""
    run_sql(f"ALTER TABLE vendor_data.{table} ADD COLUMN StockStatus VARCHAR(20)")
    run_sql(f"""
        UPDATE vendor_data.{table}
        SET StockStatus = CASE WHEN StockQuantity = 0 THEN 'Out of Stock' ELSE 'In Stock' END
    """)
    print(f'{table}: StockStatus added.')

---
## 1. begin_inventory

**Cleaning steps:**
- Drop: `InventoryId`, `startDate`
- Rename: `Brand` -> `ProductID`, `Description` -> `ProductName`, `Size` -> `BottleSize`, `Price` -> `SalesPrice`, `onHand` -> `StockQuantity`
- Cast: `SalesPrice` as DECIMAL(10,2)
- Fix: `SalesPrice = 0` for ProductID 19138 -> set to 9.99
- Add: `BottleSizeCategory`, `StockStatus`

In [11]:
drop_columns('begin_inventory', ['InventoryId', 'startDate'])

rename_columns('begin_inventory', {
    'Brand': 'ProductID',
    'Description': 'ProductName',
    'Size': 'BottleSize',
    'Price': 'SalesPrice',
    'onHand': 'StockQuantity'
})

cast_columns('begin_inventory', {'SalesPrice': 'DECIMAL(10,2)'})

# Fix SalesPrice = 0 for ProductID 19138 (Gerard Bertrand Organic Rose)
# Verified: end_inventory and purchase_prices both confirm correct price is $9.99
run_sql("""
    UPDATE vendor_data.begin_inventory
    SET SalesPrice = 9.99
    WHERE ProductID = 19138
    AND SalesPrice = 0
""")

add_bottle_size_category('begin_inventory')
add_stock_status('begin_inventory')

begin_inventory: dropped columns ['InventoryId', 'startDate']
begin_inventory: renamed columns {'Brand': 'ProductID', 'Description': 'ProductName', 'Size': 'BottleSize', 'Price': 'SalesPrice', 'onHand': 'StockQuantity'}
begin_inventory: cast columns {'SalesPrice': 'DECIMAL(10,2)'}
begin_inventory: BottleSizeCategory added.
begin_inventory: StockStatus added.


---
## 2. end_inventory

**Cleaning steps:**
- Drop: `InventoryId`, `endDate`
- Rename: same as begin_inventory
- Cast: `SalesPrice` as DECIMAL(10,2)
- Fix: Store 46 missing City -> fill with TYWARDREATH
- Add: `BottleSizeCategory`, `StockStatus`

In [12]:
drop_columns('end_inventory', ['InventoryId', 'endDate'])

rename_columns('end_inventory', {
    'Brand': 'ProductID',
    'Description': 'ProductName',
    'Size': 'BottleSize',
    'Price': 'SalesPrice',
    'onHand': 'StockQuantity'
})

cast_columns('end_inventory', {'SalesPrice': 'DECIMAL(10,2)'})

# Fix missing City for Store 46
# Verified: begin_inventory shows Store 46 = TYWARDREATH
# All null City rows belong to Store 46 only
run_sql("""
    UPDATE vendor_data.end_inventory
    SET City = 'TYWARDREATH'
    WHERE Store = 46
    AND City IS NULL
""")

add_bottle_size_category('end_inventory')
add_stock_status('end_inventory')

end_inventory: dropped columns ['InventoryId', 'endDate']
end_inventory: renamed columns {'Brand': 'ProductID', 'Description': 'ProductName', 'Size': 'BottleSize', 'Price': 'SalesPrice', 'onHand': 'StockQuantity'}
end_inventory: cast columns {'SalesPrice': 'DECIMAL(10,2)'}
end_inventory: BottleSizeCategory added.
end_inventory: StockStatus added.


---
## 3. purchase_prices -> product_master

**Cleaning steps:**
- Drop row: ProductID 4202 (broken row, never stocked anywhere)
- Rename: `Brand` -> `ProductID`, `Description` -> `ProductName`, `Size` -> `BottleSize`, `Price` -> `SalesPrice`
- Fix: `Volume = 'Unknown'` -> NULL (4 rows: ProductIDs 2993, 9908, 8992, 90590), `Volume` floating point noise -> ROUND to integer
- Fix: `VendorName` trailing whitespace
- Cast: `SalesPrice`, `PurchasePrice` as DECIMAL(10,2), `Volume` as INT UNSIGNED
- Add: `VendorCanonicalName` for vendors with legal entity name changes
- Add: `BottleSizeCategory`
- Rename: Table `purchase_prices` -> `product_master`

In [13]:
# Drop the broken row before any renaming
run_sql("DELETE FROM vendor_data.purchase_prices WHERE Brand = 4202")

rename_columns('purchase_prices', {
    'Brand': 'ProductID',
    'Description': 'ProductName',
    'Size': 'BottleSize',
    'Price': 'SalesPrice'
})

# Volume 'Unknown' -> NULL before casting to an integer type
run_sql("""
    UPDATE vendor_data.purchase_prices
    SET Volume = NULL
    WHERE Volume = 'Unknown'
""")

# Fix float noise in Volume (e.g. 750.000001) before casting to INT
run_sql("UPDATE vendor_data.purchase_prices SET Volume = ROUND(Volume, 0) WHERE Volume IS NOT NULL")

cast_columns('purchase_prices', {
    'SalesPrice': 'DECIMAL(10,2)',
    'PurchasePrice': 'DECIMAL(10,2)',
    'Volume': 'INT UNSIGNED'
})

add_vendor_canonical_name('purchase_prices')
add_bottle_size_category('purchase_prices')

# Rename table to reflect its role as a product reference/master table
run_sql("RENAME TABLE vendor_data.purchase_prices TO vendor_data.product_master")

purchase_prices: renamed columns {'Brand': 'ProductID', 'Description': 'ProductName', 'Size': 'BottleSize', 'Price': 'SalesPrice'}
purchase_prices: cast columns {'SalesPrice': 'DECIMAL(10,2)', 'PurchasePrice': 'DECIMAL(10,2)', 'Volume': 'INT UNSIGNED'}
purchase_prices: VendorName trimmed, VendorCanonicalName added.
purchase_prices: BottleSizeCategory added.


---
## 4. purchases

**Cleaning steps:**
- Drop: `InventoryId`
- Rename: `Brand` -> `ProductID`, `Description` -> `ProductName`, `Size` -> `BottleSize`, `Quantity` -> `PurchaseQuantity`, `Dollars` -> `PurchaseDollars`
- Cast: `PurchasePrice`, `PurchaseDollars` as DECIMAL(10,2)
- Fix: `VendorName` trailing whitespace
- Add: `VendorCanonicalName`, `PaymentLagDays`, `BottleSizeCategory`
- Cast: date columns (`PODate`, `ReceivingDate`, `InvoiceDate`, `PayDate`) to DATE for Power BI

In [14]:
drop_columns('purchases', ['InventoryId'])

rename_columns('purchases', {
    'Brand': 'ProductID',
    'Description': 'ProductName',
    'Size': 'BottleSize',
    'Quantity': 'PurchaseQuantity',
    'Dollars': 'PurchaseDollars'
})

cast_columns('purchases', {
    'PurchasePrice': 'DECIMAL(10,2)',
    'PurchaseDollars': 'DECIMAL(10,2)'
})

run_sql("ALTER TABLE vendor_data.purchases ADD COLUMN PaymentLagDays INT")
run_sql("""
    UPDATE vendor_data.purchases
    SET PaymentLagDays = DATEDIFF(PayDate, InvoiceDate)
""")

add_vendor_canonical_name('purchases')
add_bottle_size_category('purchases')

# Cast date columns from text to proper DATE type
# Required for Power BI time intelligence and DATEDIFF correctness
cast_columns('purchases', {
    'PODate'        : 'DATE',
    'ReceivingDate' : 'DATE',
    'InvoiceDate'   : 'DATE',
    'PayDate'       : 'DATE'
})

purchases: dropped columns ['InventoryId']
purchases: renamed columns {'Brand': 'ProductID', 'Description': 'ProductName', 'Size': 'BottleSize', 'Quantity': 'PurchaseQuantity', 'Dollars': 'PurchaseDollars'}
purchases: cast columns {'PurchasePrice': 'DECIMAL(10,2)', 'PurchaseDollars': 'DECIMAL(10,2)'}
purchases: VendorName trimmed, VendorCanonicalName added.
purchases: BottleSizeCategory added.
purchases: cast columns {'PODate': 'DATE', 'ReceivingDate': 'DATE', 'InvoiceDate': 'DATE', 'PayDate': 'DATE'}


---
## 5. sales

**Cleaning steps:**
- Drop: `InventoryId`
- Rename: `Brand` -> `ProductID`, `Description` -> `ProductName`, `Size` -> `BottleSize`, `VendorNo` -> `VendorNumber`
- Cast: `SalesDollars`, `SalesPrice`, `ExciseTax` as DECIMAL(10,2)
- Fix: `Volume` floating point noise -> ROUND to integer, then cast to INT UNSIGNED
- Fix: `VendorName` trailing whitespace
- Add: `TotalRevenue` (SalesDollars + ExciseTax), `IsFreePromotion` flag for zero-dollar promo rows, `VendorCanonicalName`, `BottleSizeCategory`

- Cast: `SalesDate` to DATE for Power BI

> **Note:** Runs on ~12.8M rows, will take a few minutes.

In [15]:
drop_columns('sales', ['InventoryId'])

rename_columns('sales', {
    'Brand': 'ProductID',
    'Description': 'ProductName',
    'Size': 'BottleSize',
    'VendorNo': 'VendorNumber'
})

cast_columns('sales', {
    'SalesDollars': 'DECIMAL(10,2)',
    'SalesPrice': 'DECIMAL(10,2)',
    'ExciseTax': 'DECIMAL(10,2)'
})

# Volume: fix floating point noise, then cast to an integer type
run_sql("UPDATE vendor_data.sales SET Volume = ROUND(Volume, 0)")
cast_columns('sales', {'Volume': 'INT UNSIGNED'})

run_sql("ALTER TABLE vendor_data.sales ADD COLUMN TotalRevenue DECIMAL(10,2)")
run_sql("UPDATE vendor_data.sales SET TotalRevenue = SalesDollars + ExciseTax")

# IsFreePromotion: real promo events (zero-dollar, real quantity), not errors - kept for vendor analysis
run_sql("ALTER TABLE vendor_data.sales ADD COLUMN IsFreePromotion TINYINT")
run_sql("""
    UPDATE vendor_data.sales
    SET IsFreePromotion = CASE
        WHEN SalesDollars = 0 AND SalesPrice = 0 AND SalesQuantity > 0 THEN 1
        ELSE 0
    END
""")

add_vendor_canonical_name('sales')
add_bottle_size_category('sales')

# Cast date column from text to proper DATE type
# Required for Power BI time intelligence
cast_columns('sales', {'SalesDate': 'DATE'})

sales: dropped columns ['InventoryId']
sales: renamed columns {'Brand': 'ProductID', 'Description': 'ProductName', 'Size': 'BottleSize', 'VendorNo': 'VendorNumber'}
sales: cast columns {'SalesDollars': 'DECIMAL(10,2)', 'SalesPrice': 'DECIMAL(10,2)', 'ExciseTax': 'DECIMAL(10,2)'}
sales: cast columns {'Volume': 'INT UNSIGNED'}
sales: VendorName trimmed, VendorCanonicalName added.
sales: BottleSizeCategory added.
sales: cast columns {'SalesDate': 'DATE'}


---
## 6. vendor_invoice

**Cleaning steps:**
- Drop: `Approval`
- Rename: `Dollars` -> `POTotalAmount`
- Cast: `POTotalAmount`, `Freight` as DECIMAL(10,2)
- Fix: `VendorName` trailing whitespace
- Add: `LandedCost` (POTotalAmount + Freight), `PaymentLagDays`, `VendorCanonicalName`
- Cast: date columns (`InvoiceDate`, `PODate`, `PayDate`) to DATE for Power BI

In [16]:
drop_columns('vendor_invoice', ['Approval'])

rename_columns('vendor_invoice', {'Dollars': 'POTotalAmount'})

cast_columns('vendor_invoice', {
    'POTotalAmount': 'DECIMAL(10,2)',
    'Freight': 'DECIMAL(10,2)'
})

run_sql("ALTER TABLE vendor_data.vendor_invoice ADD COLUMN LandedCost DECIMAL(10,2)")
run_sql("UPDATE vendor_data.vendor_invoice SET LandedCost = POTotalAmount + Freight")

run_sql("ALTER TABLE vendor_data.vendor_invoice ADD COLUMN PaymentLagDays INT")
run_sql("UPDATE vendor_data.vendor_invoice SET PaymentLagDays = DATEDIFF(PayDate, InvoiceDate)")

add_vendor_canonical_name('vendor_invoice')

# Cast date columns from text to proper DATE type
# Required for Power BI time intelligence
cast_columns('vendor_invoice', {
    'InvoiceDate' : 'DATE',
    'PODate'      : 'DATE',
    'PayDate'     : 'DATE'
})

vendor_invoice: dropped columns ['Approval']
vendor_invoice: renamed columns {'Dollars': 'POTotalAmount'}
vendor_invoice: cast columns {'POTotalAmount': 'DECIMAL(10,2)', 'Freight': 'DECIMAL(10,2)'}
vendor_invoice: VendorName trimmed, VendorCanonicalName added.
vendor_invoice: cast columns {'InvoiceDate': 'DATE', 'PODate': 'DATE', 'PayDate': 'DATE'}


##### We can perform analysis in various ways across 6 tables:

1. Vendor-Product Sales Summary — vendor profitability, margins, stock-flow ratio.
2. Procurement & Logistics Summary — freight efficiency, lead times, payment behavior.
3. Time-series analysis — seasonality, trends, promotion effects
4. Store/City performance — geographic angle
5. Inventory reconciliation — stockouts, dead stock, shrinkage, new/discontinued SKUs.
6. Dead-stock analysis — purchased-but-never-sold products.

##### As the data that we need for analysis is distributed in different tables, we need to create 4 summary tables:
  1. vendor_product_sales_summary   (purchases + sales + product_master)   grain: VendorNumber + ProductID
  2. procurement_logistics_summary  (purchases + vendor_invoice)          grain: PONumber
  3. store_city_performance_summary (purchases + sales + inventory-derived City lookup) grain: Store
  4. inventory_reconciliation_summary (begin_inventory + end_inventory + purchases + sales + product_master) grain: Store + ProductID

##### Dead-stock analysis and time-series analysis don't need new joins:
  - Dead stock  -> query vendor_product_sales_summary WHERE TotalSalesQuantity = 0
  - Time-series -> query the raw `sales` / `purchases` tables directly, grouped by date

In [1]:
# build_summary_tables.py script builds 4 persisted summary tables
# from the 6 raw tables and ingest all of them into MySQL DB for performing further data analysis and reporting work
%run build_summary_tables.py

2026-09-28 10:02:39,364 - INFO - Creating empty summary tables......
2026-09-28 10:02:40,353 - INFO - All 4 summary tables created successfully
2026-09-28 10:02:40,353 - INFO - Building vendor_product_sales_summary......
2026-09-28 10:05:30,938 - INFO - vendor_product_sales_summary built in 170.58s, 10693 rows
2026-09-28 10:05:43,312 - INFO - Loaded 'vendor_product_sales_summary': 10,693 rows in 12.37s
2026-09-28 10:05:43,320 - INFO - vendor_product_sales_summary ingested in 12.37s
2026-09-28 10:05:43,320 - INFO - Building procurement_logistics_summary......
2026-09-28 10:05:54,039 - INFO - procurement_logistics_summary built in 10.72s, 5543 rows
2026-09-28 10:05:58,784 - INFO - Loaded 'procurement_logistics_summary': 5,543 rows in 4.74s
2026-09-28 10:05:58,784 - INFO - procurement_logistics_summary ingested in 4.74s
2026-09-28 10:05:58,792 - INFO - Building store_city_performance_summary......
2026-09-28 10:07:16,900 - INFO - store_city_performance_summary built in 78.11s, 80 rows
202

Done. Succeeded: ['vendor_product_sales_summary', 'procurement_logistics_summary', 'store_city_performance_summary', 'inventory_reconciliation_summary']
